# Employee Attrition Analytics

## Module 6: Feature Engineering

### Objective

Transform the cleaned employee dataset into meaningful features suitable for machine learning.

In [1]:
import pandas as pd
import numpy as np

## Load Clean Dataset

In [2]:
df = pd.read_csv("../data/processed/employee_hr_analytics_clean.csv")
df.head()

,Employee_ID,Age,Gender,Department,Job_Role,Experience,Education,Salary,Performance_Score,Job_Satisfaction,...,Remote_Work,Training_Hours,Projects,Promotion_Last_5Yrs,Joining_Date,Attrition,City,Employment_Type,Manager_Rating,Sick_Leaves
0,1,40.0,Male,Sales,Sales Manager,3,Master,94113.0,5,6.0,...,No,70,5,No,2011-12-21,No,Rawalpindi,Contract,5,6
1,2,39.0,Female,Sales,Sales Manager,2,Master,54224.0,4,9.0,...,No,33,3,Yes,2015-09-20,No,Islamabad,Full-Time,4,4
2,3,49.0,Male,Finance,Financial Analyst,25,Master,166749.0,2,6.0,...,Yes,75,12,No,2014-12-10,No,Islamabad,Full-Time,8,2
3,4,54.0,Female,Marketing,Marketing Specialist,26,Bachelor,196590.0,5,4.0,...,Yes,64,12,No,2022-08-23,Yes,Rawalpindi,Contract,4,5
4,5,41.0,Female,IT,Data Analyst,18,PhD,231764.0,3,8.0,...,Yes,15,12,No,2021-09-29,No,Rawalpindi,Contract,10,8


## Inspect Dataset

In [3]:
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

Shape: (1000, 22)

Columns:
['Employee_ID', 'Age', 'Gender', 'Department', 'Job_Role', 'Experience', 'Education', 'Salary', 'Performance_Score', 'Job_Satisfaction', 'Overtime', 'Work_Hours', 'Remote_Work', 'Training_Hours', 'Projects', 'Promotion_Last_5Yrs', 'Joining_Date', 'Attrition', 'City', 'Employment_Type', 'Manager_Rating', 'Sick_Leaves']

Data Types:
Employee_ID              int64
Age                    float64
Gender                     str
Department                 str
Job_Role                   str
Experience               int64
Education                  str
Salary                 float64
Performance_Score        int64
Job_Satisfaction       float64
Overtime                   str
Work_Hours               int64
Remote_Work                str
Training_Hours           int64
Projects                 int64
Promotion_Last_5Yrs        str
Joining_Date               str
Attrition                  str
City                       str
Employment_Type            str
Manager_Rating     

## Convert Joining Date

In [4]:
df["Joining_Date"] = pd.to_datetime(df["Joining_Date"])

df["Joining_Date"].dtype

dtype('<M8[us]')

## Create Employee Tenure

In [6]:
reference_date = pd.Timestamp.today()

df["Years_At_Company"] = ((reference_date - df["Joining_Date"]).dt.days / 365.25).round(
    1
)

df[["Employee_ID", "Joining_Date", "Years_At_Company"]].head()

,Employee_ID,Joining_Date,Years_At_Company
0,1,2011-12-21,14.6
1,2,2015-09-20,10.9
2,3,2014-12-10,11.7
3,4,2022-08-23,4.0
4,5,2021-09-29,4.9


## Validate Employee Tenure

In [7]:
print("Minimum Tenure:", df["Years_At_Company"].min())
print("Maximum Tenure", df["Years_At_Company"].max())

Minimum Tenure: 0.0
Maximum Tenure 15.0


## Create Previous Experience Feature

In [8]:
df["Previous_Experience"] = df["Experience"] - df["Years_At_Company"]

df["Previous_Experience"] = (
    df["Previous_Experience"]
    .clip(
        lower=0,
    )
    .round(1)
)

df[["Experience", "Years_At_Company", "Previous_Experience"]].head()

,Experience,Years_At_Company,Previous_Experience
0,3,14.6,0.0
1,2,10.9,0.0
2,25,11.7,13.3
3,26,4.0,22.0
4,18,4.9,13.1


## Create Salary per Year of Experience

In [ ]:
df["Salary_Per_Experience"] = df["Salary"] / df["Experience"].replace(0, np.nan)

df["Salary_Per_Experience"] = df["Salary_Per_Experience"].fillna(df["Salary"]).round(2)

df[["Salary", "Experience", "Salary_Per_Experience"]].head()

,Salary,Experience,Salary_Per_Experience
0,94113.0,3,31371.00
1,54224.0,2,27112.00
2,166749.0,25,6669.96
3,196590.0,26,7561.15
4,231764.0,18,12875.78


## Create Salary Bands

In [10]:
df["Salary_Band"] = pd.qcut(
    df["Salary"], q=4, labels=["Low", "Medium", "High", "Very High"]
)
df["Salary_Band"].value_counts().sort_index()

Salary_Band
Low          250
Medium       262
High         238
Very High    250
Name: count, dtype: int64

## Create Experience Bands

In [12]:
df["Experience_Band"] = pd.cut(
    df["Experience"],
    bins=[-1, 2, 5, 10, np.inf],
    labels=["Entry Level", "Junior", "Mid Level", "Senior"],
)

df["Experience_Band"].value_counts()

Experience_Band
Senior         382
Entry Level    257
Mid Level      190
Junior         171
Name: count, dtype: int64

## Create Workload Indicator

In [13]:
df["Workload_Score"] = (df["Projects"] + (df["Work_Hours"] / 10)).round(1)
df[["Projects", "Work_Hours", "Workload_Score"]].head()

,Projects,Work_Hours,Workload_Score
0,5,46,9.6
1,3,36,6.6
2,12,48,16.8
3,12,51,17.1
4,12,40,16.0


## Create Satisfaction Category

In [14]:
df["Satisfaction_Category"] = pd.cut(
    df["Job_Satisfaction"], bins=[0, 4, 7, 10], labels=["Low", "Medium", "High"]
)

df["Satisfaction_Category"].value_counts()

Satisfaction_Category
Medium    476
High      363
Low       161
Name: count, dtype: int64

## Create Performance Category

In [15]:
df["Performance_Category"] = pd.cut(
    df["Performance_Score"], bins=[0, 2, 3, 5], labels=["Low", "Average", "High"]
)

df["Performance_Category"].value_counts()

Performance_Category
Low        406
High       400
Average    194
Name: count, dtype: int64

## Create Attrition Target

In [16]:
df["Attrition_Target"] = df["Attrition"].map({"Yes": 1, "No": 0})

df[["Attrition", "Attrition_Target"]].head(10)

,Attrition,Attrition_Target
0,No,0
1,No,0
2,No,0
3,Yes,1
4,No,0
5,No,0
6,Yes,1
7,No,0
8,No,0
9,No,0


## Validate Target Variable

In [17]:
print(df["Attrition_Target"].value_counts())

print("\nTarget percentages:")
print(df["Attrition_Target"].value_counts(normalize=True).mul(100).round(2))

Attrition_Target
0    882
1    118
Name: count, dtype: int64

Target percentages:
Attrition_Target
0    88.2
1    11.8
Name: proportion, dtype: float64


## Remove Employee Identifier

In [18]:
df = df.drop(columns=["Employee_ID"])

df.head()

,Age,Gender,Department,Job_Role,Experience,Education,Salary,Performance_Score,Job_Satisfaction,Overtime,...,Sick_Leaves,Years_At_Company,Previous_Experience,Salary_Per_Experience,Salary_Band,Experience_Band,Workload_Score,Satisfaction_Category,Performance_Category,Attrition_Target
0,40.0,Male,Sales,Sales Manager,3,Master,94113.0,5,6.0,Yes,...,6,14.6,0.0,31371.00,Medium,Junior,9.6,Medium,High,0
1,39.0,Female,Sales,Sales Manager,2,Master,54224.0,4,9.0,No,...,4,10.9,0.0,27112.00,Low,Entry Level,6.6,High,High,0
2,49.0,Male,Finance,Financial Analyst,25,Master,166749.0,2,6.0,No,...,2,11.7,13.3,6669.96,High,Senior,16.8,Medium,Low,0
3,54.0,Female,Marketing,Marketing Specialist,26,Bachelor,196590.0,5,4.0,Yes,...,5,4.0,22.0,7561.15,Very High,Senior,17.1,Low,High,1
4,41.0,Female,IT,Data Analyst,18,PhD,231764.0,3,8.0,Yes,...,8,4.9,13.1,12875.78,Very High,Senior,16.0,High,Average,0


## Remove Raw Joining Date

In [19]:
df = df.drop(columns=["Joining_Date"])

df.head()

,Age,Gender,Department,Job_Role,Experience,Education,Salary,Performance_Score,Job_Satisfaction,Overtime,...,Sick_Leaves,Years_At_Company,Previous_Experience,Salary_Per_Experience,Salary_Band,Experience_Band,Workload_Score,Satisfaction_Category,Performance_Category,Attrition_Target
0,40.0,Male,Sales,Sales Manager,3,Master,94113.0,5,6.0,Yes,...,6,14.6,0.0,31371.00,Medium,Junior,9.6,Medium,High,0
1,39.0,Female,Sales,Sales Manager,2,Master,54224.0,4,9.0,No,...,4,10.9,0.0,27112.00,Low,Entry Level,6.6,High,High,0
2,49.0,Male,Finance,Financial Analyst,25,Master,166749.0,2,6.0,No,...,2,11.7,13.3,6669.96,High,Senior,16.8,Medium,Low,0
3,54.0,Female,Marketing,Marketing Specialist,26,Bachelor,196590.0,5,4.0,Yes,...,5,4.0,22.0,7561.15,Very High,Senior,17.1,Low,High,1
4,41.0,Female,IT,Data Analyst,18,PhD,231764.0,3,8.0,Yes,...,8,4.9,13.1,12875.78,Very High,Senior,16.0,High,Average,0


## Encode Binary Features

In [ ]:
binary_mappings = {
    "Gender": {"Male": 0, "Female": 1},
    "Overtime": {"No": 0, "Yes": 1},
    "Remote_Work": {"No": 0, "Yes": 1},
}

for column, mapping in binary_mappings.items():
    df[column] = df[column].map(mapping)

## One-Hot Encode Categorical Variables

In [ ]:
categorical_columns = [
    "Department",
    "Job_Role",
    "Education",
    "City",
    "Employment_Type",
    "Salary_Band",
    "Experience_Band",
    "Satisfaction_Category",
    "Performance_Category",
]

df = pd.get_dummies(df, columns=categorical_columns, drop_first=True, dtype=int)

df.head()

,Age,Gender,Experience,Salary,Performance_Score,Job_Satisfaction,Overtime,Work_Hours,Remote_Work,Training_Hours,...,Salary_Band_Medium,Salary_Band_High,Salary_Band_Very High,Experience_Band_Junior,Experience_Band_Mid Level,Experience_Band_Senior,Satisfaction_Category_Medium,Satisfaction_Category_High,Performance_Category_Average,Performance_Category_High
0,40.0,0,3,94113.0,5,6.0,1,46,0,70,...,1,0,0,1,0,0,1,0,0,1
1,39.0,1,2,54224.0,4,9.0,0,36,0,33,...,0,0,0,0,0,0,0,1,0,1
2,49.0,0,25,166749.0,2,6.0,0,48,1,75,...,0,1,0,0,0,1,1,0,0,0
3,54.0,1,26,196590.0,5,4.0,1,51,1,64,...,0,0,1,0,0,1,0,0,0,1
4,41.0,1,18,231764.0,3,8.0,1,40,1,15,...,0,0,1,0,0,1,0,1,1,0


## Inspect ML Features

In [22]:
print("Final dataset shape:", df.shape)

print("\nFinal columns:")
print(df.columns.tolist())

Final dataset shape: (1000, 52)

Final columns:
['Age', 'Gender', 'Experience', 'Salary', 'Performance_Score', 'Job_Satisfaction', 'Overtime', 'Work_Hours', 'Remote_Work', 'Training_Hours', 'Projects', 'Promotion_Last_5Yrs', 'Attrition', 'Manager_Rating', 'Sick_Leaves', 'Years_At_Company', 'Previous_Experience', 'Salary_Per_Experience', 'Workload_Score', 'Attrition_Target', 'Department_HR', 'Department_IT', 'Department_Marketing', 'Department_Sales', 'Job_Role_Data Analyst', 'Job_Role_DevOps Engineer', 'Job_Role_Financial Analyst', 'Job_Role_HR Executive', 'Job_Role_Marketing Specialist', 'Job_Role_Recruiter', 'Job_Role_SEO Executive', 'Job_Role_Sales Executive', 'Job_Role_Sales Manager', 'Job_Role_Software Engineer', 'Education_Master', 'Education_PhD', 'City_Islamabad', 'City_Karachi', 'City_Lahore', 'City_Peshawar', 'City_Rawalpindi', 'Employment_Type_Full-Time', 'Salary_Band_Medium', 'Salary_Band_High', 'Salary_Band_Very High', 'Experience_Band_Junior', 'Experience_Band_Mid Level',

## Validate Data Types

In [23]:
print(df.dtypes.value_counts())

int64      43
float64     7
str         2
Name: count, dtype: int64


## Check Missing Values

In [24]:
missing_values = df.isnull().sum()

print("Total missing values:", missing_values.sum())

Total missing values: 0


## Check Duplicate Rows

In [25]:
print(
    "Duplicate rows:",
    df.duplicated().sum()
)

Duplicate rows: 0


## Separate Features and Target

In [27]:
X = df.drop(
    columns=["Attrition", "Attrition_Target"]
)

y = df["Attrition_Target"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (1000, 50)
Target shape: (1000,)


## Validate Feature Matrix

In [28]:
print("Number of features:", X.shape[1])

print(
    "All features numeric:",
    X.select_dtypes(exclude=np.number).shape[1] == 0
)

print(
    "Missing feature values:",
    X.isnull().sum().sum()
)

Number of features: 50
All features numeric: False
Missing feature values: 0


## Save ML-Ready Dataset

In [29]:
ml_dataset_path = (
    "../data/processed/"
    "employee_attrition_ml_ready.csv"
)

df.to_csv(
    ml_dataset_path,
    index=False
)

print(
    f"ML-ready dataset saved to: {ml_dataset_path}"
)

ML-ready dataset saved to: ../data/processed/employee_attrition_ml_ready.csv


## Save Features and Target

In [30]:
X.to_csv(
    "../data/processed/employee_features.csv",
    index=False
)

y.to_csv(
    "../data/processed/employee_target.csv",
    index=False
)

print("Features and target saved successfully.")

Features and target saved successfully.


## Final Feature Engineering Validation

In [31]:
print("Final ML Dataset")
print("=" * 40)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nFeatures:", X.shape[1])

print("\nTarget Distribution:")
print(y.value_counts())

print("\nMissing Values:", df.isnull().sum().sum())

print("Duplicate Rows:", df.duplicated().sum())

Final ML Dataset
Rows: 1000
Columns: 52

Features: 50

Target Distribution:
Attrition_Target
0    882
1    118
Name: count, dtype: int64

Missing Values: 0
Duplicate Rows: 0
